# GBM 하이퍼파라미터 random search (LightGBM)

현행 파라미터(트리600·lr0.03·leaves63)는 한 번도 탐색 안 됨. 24개 설정을 **GBM 단독 예측**의 2폴드 공식 점수(contrib=FICR−NMAE)로 평가.
채택: **두 해 모두 현행 이상** 중 worst-year 최대. torch 미사용 → 멀티스레드.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import lightgbm as lgb
import wind_lib as W
from official_metric import group_scores
np.random.seed(7)

GROUPS=(1,2,3); FR,TGT={},{}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
FEATS=W.lean_features(BASE_ALL)+W.SPATIAL_COLS
FOLDS={2023:[2022],2024:[2022,2023]}

# 폴드 프레임 캐시 (파워커브 포함)
CACHE={}
for vy,tys in FOLDS.items():
    ent={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        tr=fr[yr.isin(tys)]; va=fr[yr==vy]
        if len(tr)==0 or len(va)==0: continue
        iso=W.fit_powercurve(tr,tgt,cap)
        ent[g]=(W.with_pc(tr,iso),W.with_pc(va,iso))
    CACHE[vy]=ent

BASE_PARAMS=dict(objective="mae",n_estimators=600,learning_rate=0.03,num_leaves=63,
    min_child_samples=60,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,reg_lambda=1.0,
    random_state=42,n_jobs=-1,verbose=-1)

def eval_params(params):
    out={}
    for vy,ent in CACHE.items():
        nm=[]; fi=[]
        for g,(tr2,va2) in ent.items():
            tgt=TGT[g]; cap=W.CAP[g]
            m=lgb.LGBMRegressor(**params).fit(tr2[FEATS],tr2[tgt])
            p=np.clip(m.predict(va2[FEATS]),0,cap)
            a,b=group_scores(va2[tgt].to_numpy(),p,cap); nm.append(a); fi.append(b)
        out[vy]=0.5*(1-np.mean(nm))+0.5*np.mean(fi)
    return out

base=eval_params(BASE_PARAMS)
print(f"현행: 2023={base[2023]:.4f}  2024={base[2024]:.4f}")

rng=np.random.RandomState(7)
def sample():
    p=dict(BASE_PARAMS)
    p.update(n_estimators=int(rng.choice([400,800,1200,2000])),
             learning_rate=float(10**rng.uniform(np.log10(0.01),np.log10(0.08))),
             num_leaves=int(rng.choice([31,63,127,255])),
             min_child_samples=int(rng.choice([20,60,150,300])),
             colsample_bytree=float(rng.choice([0.5,0.7,0.9])),
             subsample=float(rng.choice([0.7,0.8,1.0])),
             reg_lambda=float(rng.choice([0.1,1.0,10.0])))
    return p

results=[]
for t in range(24):
    p=sample(); r=eval_params(p)
    ok=(r[2023]>=base[2023]) and (r[2024]>=base[2024])
    results.append((t,p,r,ok))
    print(f"trial {t:2d}: 2023={r[2023]:.4f} 2024={r[2024]:.4f} min={min(r.values()):.4f} "
          f"lr={p['learning_rate']:.3f} nl={p['num_leaves']} ne={p['n_estimators']} mcs={p['min_child_samples']} beats={ok}")

cands=[x for x in results if x[3]]
if cands:
    t,bp,br,_=max(cands,key=lambda x:min(x[2].values()))
    print(f"\n채택 trial {t}: 2023={br[2023]:.4f} 2024={br[2024]:.4f} (현행 {base[2023]:.4f}/{base[2024]:.4f})")
    save={k:v for k,v in bp.items() if k not in ("n_jobs",)}
else:
    print("\n현행을 두 해 모두 이기는 설정 없음 → 현행 유지")
    save={k:v for k,v in BASE_PARAMS.items() if k not in ("n_jobs",)}; br=base; t=None
json.dump(dict(adopted_trial=t, params=save,
               totals={"base":{str(k):round(v,4) for k,v in base.items()},
                       "best":{str(k):round(v,4) for k,v in br.items()}}),
          open("submission/ver_7/gbm_hpo_result.json","w"),ensure_ascii=False,indent=2)
print("saved gbm_hpo_result.json")

현행: 2023=0.5886  2024=0.6008


trial  0: 2023=0.5914 2024=0.6064 min=0.5914 lr=0.016 nl=127 ne=2000 mcs=300 beats=True


trial  1: 2023=0.5918 2024=0.6013 min=0.5918 lr=0.028 nl=127 ne=2000 mcs=150 beats=True


trial  2: 2023=0.5854 2024=0.6006 min=0.5854 lr=0.061 nl=255 ne=400 mcs=20 beats=False


trial  3: 2023=0.5870 2024=0.6000 min=0.5870 lr=0.042 nl=255 ne=2000 mcs=60 beats=False


...

trial 19: 2023=0.5893 2024=0.6031 min=0.5893 lr=0.015 nl=63 ne=1200 mcs=150 beats=True


trial 20: 2023=0.5903 2024=0.6020 min=0.5903 lr=0.022 nl=31 ne=2000 mcs=60 beats=True


trial 21: 2023=0.5869 2024=0.6013 min=0.5869 lr=0.023 nl=255 ne=2000 mcs=60 beats=False


trial 22: 2023=0.5873 2024=0.6004 min=0.5873 lr=0.021 nl=63 ne=2000 mcs=60 beats=False


trial 23: 2023=0.5817 2024=0.5976 min=0.5817 lr=0.011 nl=255 ne=400 mcs=60 beats=False

채택 trial 7: 2023=0.5918 2024=0.6049 (현행 0.5886/0.6008)
saved gbm_hpo_result.json
